# Reliability and Reconciliation (Notebook 05)

Two coders labelled a subset of the audit sample independently:

- **Coder 1** — the full 88 episodes, coded from the **English** (human translation).
- **Coder 2** — 43 of the 88, coded from the **original Polish**.

This notebook computes, reproducibly from the files: (1) first-pass **inter-coder
agreement** on the double-coded subset, (2) the **adjudication** of every
`is_contestation` disagreement into a reconciled label, (3) the **reconciled gold
standard**, and (4) the **gate** re-checked on that reconciled gold. It also
categorises the *sources* of disagreement from the adjudicator's notes.

In [1]:
# ============================================================
# 1. Config, paths, load the three coding artefacts
# ============================================================
from __future__ import annotations
import json, logging, re
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
LOGGER = logging.getLogger("xai_contestation")

def find_project_root(start=None):
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for c in [start, *start.parents]:
        if (c / "data").is_dir(): return c
    raise FileNotFoundError("project root (a parent with data/) not found")
ROOT = find_project_root()
TABLES = ROOT / "outputs" / "02_translation_and_feasibility_audit" / "tables"
OUT = ROOT / "outputs" / "05_reliability_and_reconciliation"; OUT.mkdir(parents=True, exist_ok=True)

DIMS = ["presence", "target", "interaction_act", "grounds", "expected_response"]
def norm(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return pd.NA
    s = re.sub(r"\s+", " ", str(v).strip().lower())
    return pd.NA if s in ("", "nan") else s

coder1 = pd.read_csv(TABLES / "feasibility_review_sample_HUMAN.csv")            # English, 88
coder2 = pd.read_excel(TABLES / "feasibility_review_sample_CODING copy.xlsx",
                       sheet_name="Coding", engine="openpyxl")                   # Polish, 43
adj = pd.read_excel(TABLES / "disagreements_for_adjudication.xlsx",
                    sheet_name="Disagreements", engine="openpyxl")               # 15 resolved
for df in (coder1, coder2):
    for c in ["is_contestation"] + DIMS:
        if c in df.columns: df[c] = df[c].map(norm)
for c in ["C1_is_contestation", "C2_is_contestation", "ADJUDICATED_is_contestation"]:
    adj[c] = adj[c].map(norm)
LOGGER.info("coder1 %d rows | coder2 %d labelled | adjudicated %d",
            len(coder1), int(coder2["is_contestation"].notna().sum()),
            int(adj["ADJUDICATED_is_contestation"].notna().sum()))

/Users/sabrimanai/software/uj/detecting-contestation-xai/.venv/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
INFO | coder1 88 rows | coder2 43 labelled | adjudicated 15


In [2]:
# ============================================================
# 2. First-pass inter-coder agreement (independent double-coding)
# ============================================================
def cohens_kappa(a, b):
    cats = sorted(set(a) | set(b)); idx = {c: i for i, c in enumerate(cats)}
    M = np.zeros((len(cats), len(cats)))
    for x, y in zip(a, b): M[idx[x], idx[y]] += 1
    tot = M.sum(); po = np.trace(M) / tot
    pe = sum((M[i, :].sum() / tot) * (M[:, i].sum() / tot) for i in range(len(cats)))
    return (po - pe) / (1 - pe) if pe != 1 else 1.0

g = coder1.set_index("__utt_id"); s = coder2.set_index("__utt_id")
both = [u for u in g.index if u in s.index
        and pd.notna(g.loc[u, "is_contestation"]) and pd.notna(s.loc[u, "is_contestation"])]
a = [g.loc[u, "is_contestation"] for u in both]
b = [s.loc[u, "is_contestation"] for u in both]
raw = float(np.mean([x == y for x, y in zip(a, b)]))
kappa = cohens_kappa(a, b)
print(f"double-coded episodes: {len(both)}")
print(f"is_contestation  raw agreement = {raw:.0%}   Cohen's kappa = {kappa:.2f}  (first pass, pre-reconciliation)")
print("\nconfusion (rows = Coder 1 / English, cols = Coder 2 / Polish):")
display(pd.crosstab(pd.Series(a, name="C1 English"), pd.Series(b, name="C2 Polish")))
first_pass = {"n_double_coded": len(both), "raw_agreement": round(raw, 3),
              "cohens_kappa": round(kappa, 3)}

double-coded episodes: 43
is_contestation  raw agreement = 65%   Cohen's kappa = 0.30  (first pass, pre-reconciliation)

confusion (rows = Coder 1 / English, cols = Coder 2 / Polish):


C2 Polish,no,yes
C1 English,,
no,17,5
yes,10,11


In [3]:
# ============================================================
# 3. Categorise the sources of disagreement (adjudicator notes)
# ============================================================
def bucket(note):
    t = str(note).lower()
    if any(k in t for k in ["interviewer", "speaker"]):        return "speaker/interviewer artifact"
    if "context not on text" in t or "target" in t:            return "unit rule (context vs target)"
    if any(k in t for k in ["hidden", "not obvious", "context gave", "not enough context",
                            "didn't have much context", "missing data", "misinterpret",
                            "missinterpret", "make much sense", "make sense"]):
        return "implicit / hidden contestation"
    if "changed my mind" in t or "reviewing" in t:             return "coder revised on review"
    return "other / genuine ambiguity"

adj["source"] = adj["resolution_notes"].map(bucket)
print("disagreement sources (n=15):")
display(adj["source"].value_counts().rename("count").to_frame())
print("adjudication matched Coder 1:", int((adj["ADJUDICATED_is_contestation"]==adj["C1_is_contestation"]).sum()),
      "| matched Coder 2:", int((adj["ADJUDICATED_is_contestation"]==adj["C2_is_contestation"]).sum()))
adj[["__utt_id","speaker","C1_is_contestation","C2_is_contestation",
     "ADJUDICATED_is_contestation","source","resolution_notes"]].to_csv(OUT/"disagreement_sources.csv", index=False)

disagreement sources (n=15):


,count
source,
implicit / hidden contestation,10
coder revised on review,3
unit rule (context vs target),1
speaker/interviewer artifact,1


adjudication matched Coder 1: 9 | matched Coder 2: 6


In [4]:
# ============================================================
# 4. Apply adjudication -> reconciled gold standard
# ============================================================
recon = coder1.copy().set_index("__utt_id")
adj_map = adj.set_index("__utt_id")["ADJUDICATED_is_contestation"].to_dict()
c2i = coder2.set_index("__utt_id")

flip_yes_no, flip_no_yes, need_check = [], [], []
for u, new in adj_map.items():
    old = recon.loc[u, "is_contestation"]
    recon.loc[u, "is_contestation"] = new
    if old == "yes" and new == "no":              # clear structure fields
        recon.loc[u, "presence"] = "absent"
        for d in ["target","interaction_act","grounds","expected_response"]:
            recon.loc[u, d] = "n/a"
        flip_yes_no.append(u)
    elif old == "no" and new == "yes":            # seed structure from Coder 2 (who said yes)
        flip_no_yes.append(u)
        for d in DIMS:
            v = c2i.loc[u, d] if (u in c2i.index and d in c2i.columns) else pd.NA
            recon.loc[u, d] = v
        if any(pd.isna(recon.loc[u, d]) for d in DIMS):
            need_check.append(u)

recon = recon.reset_index()
recon.to_csv(TABLES / "feasibility_review_sample_GOLD.csv", index=False)   # the reconciled gold
print(f"reconciled: {len(flip_yes_no)} flipped yes->no, {len(flip_no_yes)} flipped no->yes")
print("flip yes->no:", flip_yes_no)
print("flip no->yes:", flip_no_yes)
if need_check:
    print("\n!! these no->yes rows still need taxonomy fields filled by a human:", need_check)
print("\nwrote reconciled gold -> feasibility_review_sample_GOLD.csv")

reconciled: 2 flipped yes->no, 4 flipped no->yes
flip yes->no: ['PK_DE_04:162', 'DR_SSH_02:283']
flip no->yes: ['MZ_SSH_02:273', 'MW_SSH_05:122', 'PK_DE_09:53', 'PK_DE_04:132']

wrote reconciled gold -> feasibility_review_sample_GOLD.csv


In [5]:
# ============================================================
# 5. Re-check the feasibility gate on the reconciled gold
# ============================================================
ic = recon["is_contestation"]
n = len(recon); n_yes = int((ic == "yes").sum()); n_no = int((ic == "no").sum())
precision = n_yes / n
yes = recon[ic == "yes"]
groups = sorted(recon["participant_group"].dropna().unique())
formats = [f for f in ["Descriptive statistics","SHAP","LIME","Anchor","Counterfactual"]
           if f in set(recon["explanation_format"])]
gxf = (pd.crosstab(yes["participant_group"], yes["explanation_format"])
       .reindex(index=groups, columns=formats, fill_value=0))
print(f"reconciled gold: {n} episodes | contestation = {n_yes} ({precision:.0%}) | non = {n_no}")
print(f"groups {yes['participant_group'].nunique()}/{len(groups)}  formats {yes['explanation_format'].nunique()}/{len(formats)}")
display(gxf.assign(Total=gxf.sum(axis=1)))
gxf.to_csv(OUT / "confirmed_contestation_group_by_format_reconciled.csv")

summary = {
    "first_pass_agreement": first_pass,
    "adjudicated": {"yes": int((adj["ADJUDICATED_is_contestation"]=="yes").sum()),
                    "no": int((adj["ADJUDICATED_is_contestation"]=="no").sum()),
                    "matched_coder1": int((adj["ADJUDICATED_is_contestation"]==adj["C1_is_contestation"]).sum()),
                    "matched_coder2": int((adj["ADJUDICATED_is_contestation"]==adj["C2_is_contestation"]).sum())},
    "disagreement_sources": adj["source"].value_counts().to_dict(),
    "reconciled_gold": {"n": n, "contestation": n_yes, "precision": round(precision, 3),
                        "groups": f"{yes['participant_group'].nunique()}/{len(groups)}",
                        "formats": f"{yes['explanation_format'].nunique()}/{len(formats)}"},
    "verdict": ("KEEP contestation as the central construct"
                if (yes['participant_group'].nunique()==len(groups)
                    and yes['explanation_format'].nunique()==len(formats) and n_yes>=20)
                else "reconsider scope"),
}
(OUT / "reliability_summary.json").write_text(json.dumps(summary, indent=2))
print("\nverdict:", summary["verdict"])
print("wrote reliability_summary.json")

reconciled gold: 88 episodes | contestation = 49 (56%) | non = 39
groups 3/3  formats 5/5


explanation_format,Descriptive statistics,SHAP,LIME,Anchor,Counterfactual,Total
participant_group,,,,,,
DE,4,4,2,4,4,18
IT,5,2,2,1,3,13
SSH,4,5,2,3,4,18



verdict: KEEP contestation as the central construct
wrote reliability_summary.json


## What this feeds in the paper

- **First-pass agreement** (Cohen's κ on the double-coded subset) → `sec:reliability` /
  `tab:irr`, reported honestly with the caveat that Coder 1 read the English
  translation and Coder 2 the original Polish.
- **Disagreement sources** (`disagreement_sources.csv`) → the reliability discussion
  and the error-analysis: most disagreements are *implicit / hidden* contestation,
  which motivates the "reliability stats can misfit interpretive coding" point and
  the detection error-analysis; one was a materials artifact (Coder 2's file lacked
  speaker labels); one was the target-vs-context unit rule.
- **`feasibility_review_sample_GOLD.csv`** is now the **reconciled gold standard** —
  it supersedes the single-coder file for the benchmark and all reported counts.
- The **gate still passes** on the reconciled gold: contestation stays the construct.

First-pass κ is modest — expected for an implicit, interpretive phenomenon on a
first independent pass. Report it plainly, show the sources, and lean the rigour
argument on the codebook + adjudication.